<a href="https://colab.research.google.com/github/Ayushi-369/FlyrankAi-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayushi-369/FlyrankAi-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The Rule: If a page is older than 365 days (Stale) AND sits on Page 1 (avg_position <= 10) but has a terrible CTR (< 2%), flag it as top priority (Score: 90). If it's just very stale (> 500 days), give it medium priority (Score: 60).

Reason Codes: Stale + Page 1 Poor CTR, Very Stale, Page 1 Poor CTR.

In [3]:
import os
import pandas as pd

# Load dataset safely
path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(path)

print("--- SIGNAL 1: Staleness (content_age_days) ---")
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, labels=['New', 'Mid', 'Old', 'Very Old'])
signal1 = df.groupby('age_bucket', observed=False).agg(n=('content_id', 'count'))
print(signal1)
print("Verdict: MIXED (Older content has a higher volume, but age alone is a blunt instrument.)\n")

print("--- SIGNAL 2: CTR vs Position (Page 1 with CTR < 2%) ---")
df['page_1_poor_ctr'] = ((df['avg_position'] <= 10) & (df['ctr'] < 0.02)).astype(int)
signal2 = df.groupby('page_1_poor_ctr', observed=False).agg(n=('content_id', 'count'))
print(signal2)
print("Verdict: CONFIRMED (Pages on page 1 with poor CTR have a significantly high occurrence.)")

--- SIGNAL 1: Staleness (content_age_days) ---
               n
age_bucket      
New         7518
Mid         8128
Old         6917
Very Old    7437
Verdict: MIXED (Older content has a higher volume, but age alone is a blunt instrument.)

--- SIGNAL 2: CTR vs Position (Page 1 with CTR < 2%) ---
                     n
page_1_poor_ctr       
0                24147
1                 5853
Verdict: CONFIRMED (Pages on page 1 with poor CTR have a significantly high occurrence.)


In [2]:
print('DataFrame columns:', df.columns.tolist())

DataFrame columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'age_bucket']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Encode the rule
def assign_score(row):
    if row['content_age_days'] > 365 and row['avg_position'] <= 10 and row['ctr'] < 0.02:
        return pd.Series([90, "Stale + Page 1 Poor CTR", "Queue for Rewrite"])
    elif row['content_age_days'] > 500:
        return pd.Series([60, "Very Stale", "Queue for Light Update"])
    elif row['avg_position'] <= 10 and row['ctr'] < 0.02:
        return pd.Series([50, "Page 1 Poor CTR", "Review Meta Tags"])
    return pd.Series([0, "Healthy", "Do Nothing"])

df[['action_score', 'reason_code', 'action_label']] = df.apply(assign_score, axis=1)

# 2. Rank the queue
df_ranked = df.sort_values(by='action_score', ascending=False)

# 3. Write to CSV
output_dir = "../../work/outputs"
os.makedirs(output_dir, exist_ok=True) # Create folder if in Colab environment
csv_path = f"{output_dir}/baseline_action_score.csv"
df_ranked.to_csv(csv_path, index=False)

print(f"✅ Ranked queue built! Top score is {df_ranked['action_score'].max()}.")
print(f"✅ Saved to {csv_path} (Will be ignored by git via .gitignore)")

✅ Ranked queue built! Top score is 90.
✅ Saved to ../../work/outputs/baseline_action_score.csv (Will be ignored by git via .gitignore)


## 3. Top-10 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Review: For the top 10 rows flagged with "Stale + Page 1 Poor CTR", the action is to send to editorial for a rewrite.
What would make it wrong: If the page targets a keyword with inherently low CTR (like a quick-answer intent query where users don't click), rewriting it wastes money because the 1% CTR is actually normal for that specific SERP.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("--- TOP 10 PICKS ---")
top_10 = df_ranked[df_ranked['action_score'] > 0].head(10)
display_cols = ['content_id', 'action_score', 'reason_code', 'content_age_days', 'avg_position', 'ctr', 'impressions_90d']
print(top_10[display_cols].to_string(index=False))

--- TOP 10 PICKS ---
          content_id  action_score             reason_code  content_age_days  avg_position  ctr  impressions_90d
content_aa42e8c48d7b            90 Stale + Page 1 Poor CTR               374           6.0  0.0                2
content_0065d16af779            90 Stale + Page 1 Poor CTR               460           8.9  0.0               11
content_fd3a9777ca26            90 Stale + Page 1 Poor CTR               460           4.5  0.0                2
content_cdcc4a978ac0            90 Stale + Page 1 Poor CTR               445           6.2  0.0              262
content_86cfab0defb4            90 Stale + Page 1 Poor CTR               419           9.7  0.0              375
content_a160323051fe            90 Stale + Page 1 Poor CTR               502           8.8  0.0               12
content_e5a8021fc0d9            90 Stale + Page 1 Poor CTR               460           7.8  0.0                6
content_5f68880b2920            90 Stale + Page 1 Poor CTR               46

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Pages with Score 60 ("Very Stale") often have high needs_refresh = 0 (False Positives). Age does not equal decay if the content is evergreen.
Leakage Check: Confirmed that impressions_next_30d or the needs_refresh label itself were strictly excluded from the assign_score logic.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Leakage Check Verification:")
used_features = ['content_age_days', 'avg_position', 'ctr']
leaky_features = [f for f in used_features if 'next_' in f or f == target_col]
print(f"Leaky features used in rule: {leaky_features}")
if len(leaky_features) == 0:
    print("✅ Clear: No future data or target labels leaked into the baseline rule.")

Leakage Check Verification:
Leaky features used in rule: []
✅ Clear: No future data or target labels leaked into the baseline rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.